# csv

对应 `stdlib.md`：读写 CSV。

笔记本在 `python_base/csv/qa.ipynb`。需要读写文件时，工作空间就是这个目录，题目文件落在旁边的子目录里。

先运行下一格，得到 `ROOT`。每题只改 `# 作答` 下面的代码。前置代码不用改。做完自己跑通即可，先不要对答案。


In [1]:
from pathlib import Path

def lab_root() -> Path:
    """qa.ipynb 所在目录，即 python_base/csv。"""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "csv" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "python_base" / "csv"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    return here

ROOT = lab_root()
ROOT


PosixPath('/Users/keyficller/Documents/AEFS-Notes/codes/python_base/csv')

## 1. 按表头读成字典

`people.csv` 第一行是表头。逐行读成字典，按文件顺序打印每个 `name`。


In [4]:
box = ROOT / "q1"
box.mkdir(parents=True, exist_ok=True)
path = box / "people.csv"
path.write_text("name,score\n小明,90\n小红,75\n", encoding="utf-8")

# 作答

import csv

with path.open("r", encoding="utf-8") as f:
    header = next(csv.reader(f))
    for row in csv.reader(f):
        print(f"name: {row[header.index('name')]}")
#评阅
# 名字顺序对了，但题目要「读成字典」。用 csv.DictReader，每行直接是 dict，取 row["name"]。

#参考答案
# import csv
# with path.open(encoding="utf-8", newline="") as f:
#     for row in csv.DictReader(f):
#         print(row["name"])


name: 小明
name: 小红


## 2. 带表头写出去

把 `rows` 写进 `box/out.csv`。第一行是表头 `name,score`，后面两行是数据。写完再读文件全文并打印。


In [11]:
box = ROOT / "q2"
box.mkdir(parents=True, exist_ok=True)
path = box / "out.csv"
path.unlink(missing_ok=True)
rows = [
    {"name": "小明", "score": "90"},
    {"name": "小红", "score": "75"},
]

# 作答

import csv

with path.open("w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(rows[0].keys())
    for row in rows:
        writer.writerow(row.values())

for line in path.read_text(encoding="utf-8").splitlines():
    print(line)
#评阅
# 对。表头和两行数据都写对了，也打印了全文。有现成字典时更常写 DictWriter：writeheader + writerows。

#参考答案
# import csv
# with path.open("w", encoding="utf-8", newline="") as f:
#     writer = csv.DictWriter(f, fieldnames=["name", "score"])
#     writer.writeheader()
#     writer.writerows(rows)
# print(path.read_text(encoding="utf-8"), end="")


name,score
小明,90
小红,75


## 3. 读成列表行

不要按表头读。把 `raw.csv` 每一行读成列表，排序后打印所有行。


In [12]:
box = ROOT / "q3"
box.mkdir(parents=True, exist_ok=True)
path = box / "raw.csv"
path.write_text("b,2\na,1\n", encoding="utf-8")

# 作答

import csv

for line in path.read_text(encoding="utf-8").splitlines():
    print(line)
#评阅
# 不对。要用 csv.reader 读成列表，再排序后打印。现在是整行字符串，也没排序（应先 a,1 再 b,2）。

#参考答案
# import csv
# with path.open(encoding="utf-8", newline="") as f:
#     rows = list(csv.reader(f))
# for row in sorted(rows):
#     print(row)


b,2
a,1


## 4. 分号分隔

`semi.csv` 用分号隔开字段。按表头读成字典，打印每个 `city`。


In [14]:
box = ROOT / "q4"
box.mkdir(parents=True, exist_ok=True)
path = box / "semi.csv"
path.write_text("city;pop\n上海;2400\n杭州;1200\n", encoding="utf-8")

# 作答

import csv

with path.open("r", encoding="utf-8") as f:
    header = next(csv.reader(f, delimiter=";"))
    for row in csv.reader(f, delimiter=";"):
        print(f"city: {row[header.index('city')]}")
#评阅
# city 对了，但题目要按表头读成字典。用 DictReader(..., delimiter=";")，取 row["city"]。

#参考答案
# import csv
# with path.open(encoding="utf-8", newline="") as f:
#     for row in csv.DictReader(f, delimiter=";"):
#         print(row["city"])


city: 上海
city: 杭州


## 5. 字段里有逗号

`quoted.csv` 里有的字段本身含逗号。按表头读成字典，打印每个 `note`。


In [ ]:
box = ROOT / "q5"
box.mkdir(parents=True, exist_ok=True)
path = box / "quoted.csv"
path.write_text('name,note\n小明,"你好,世界"\n小红,ok\n', encoding="utf-8")

# 作答

import csv

with path.open("r", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        print(f"note: {row['note']}")
#评阅
# note 含逗号也读对了。题目要读成字典，惯用法是 DictReader，它会自己处理引号字段。

#参考答案
# import csv
# with path.open(encoding="utf-8", newline="") as f:
#     for row in csv.DictReader(f):
#         print(row["note"])


note: 你好,世界
note: ok


## 6. 筛一行再写出

从 `people.csv` 按表头读入。只保留 `score` 转成整数后大于等于 `80` 的行，写进 `passed.csv`（同样带表头）。打印 `passed.csv` 全文。


In [20]:
box = ROOT / "q6"
box.mkdir(parents=True, exist_ok=True)
src = box / "people.csv"
src.write_text("name,score\n小明,90\n小红,75\n小刚,88\n", encoding="utf-8")
dst = box / "passed.csv"
dst.unlink(missing_ok=True)

# 作答

import csv

header : list[str] = []
rows : list[list[str]] = []
with src.open("r", encoding="utf-8") as f:
    header = next(csv.reader(f))
    rows = [row for row in csv.reader(f)]

    rows = [row for row in rows if int(row[header.index("score")]) >= 80]

with dst.open("w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerows(rows)

for line in dst.read_text(encoding="utf-8").splitlines():
    print(line)
#评阅
# 对。筛掉 75，留下 90 和 88，表头也写回去了。

#参考答案
# import csv
# with src.open(encoding="utf-8", newline="") as f:
#     rows = [row for row in csv.DictReader(f) if int(row["score"]) >= 80]
# with dst.open("w", encoding="utf-8", newline="") as f:
#     writer = csv.DictWriter(f, fieldnames=["name", "score"])
#     writer.writeheader()
#     writer.writerows(rows)
# print(dst.read_text(encoding="utf-8"), end="")


name,score
小明,90
小刚,88
